# DoRA Reimplementation: LLaMA-13B


## 1. Environment Setup

In [ ]:
!pip install -q diffusers transformers accelerate datasets

In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)

In [ ]:
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from typing import Iterable, Tuple

from diffusers import StableDiffusionXLPipeline, DDPMScheduler, AutoencoderKL, UNet2DConditionModel
from transformers import AutoTokenizer, PretrainedConfig, CLIPTextModel, CLIPTextModelWithProjection

print("Environment setup complete. All libraries imported successfully.")

## 2. Configuration

Point `DORA_ROOT` to the folder containing `datasets/`. In Colab, upload/copy the Desktop `DoRA` folder to `/content/DoRA` or Google Drive and edit the path below. Set `BASE_MODEL` to your accessible LLaMA-7B Hugging Face path or local checkpoint.

In [ ]:
# Model and Dataset paths
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
DATASET_ID = "diffusers/dog-example"
INSTANCE_PROMPT = "a photo of sks dog"

# DoRA Hyperparameters
RANK = 8
ALPHA = 16
DROPOUT = 0.0

# Training Hyperparameters
BATCH_SIZE = 1
STEPS = 500  # 500 is enough for a quick sanity check. Use 1000+ for actual quality.
LR = 1e-4

# Hardware Configurations
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Automatically use bfloat16 if supported by GPU (e.g., A100, H100, RTX 30/40 series)
WEIGHT_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32

print(f"Configuration loaded.")
print(f"Running on DEVICE: {DEVICE} | Precision: {WEIGHT_DTYPE}")

### Masking Smoketest

## 3. DoRA Layer

For a frozen pretrained weight `W0`, DoRA trains a magnitude vector `m` and low-rank direction update `BA`:

$$W' = m \frac{W_0 + BA}{\lVert W_0 + BA \rVert_c}$$

PyTorch `Linear` stores weights as `(out_features, in_features)`, so this implementation normalizes each output row. This is equivalent to column-wise normalization under the transposed convention used by `F.linear`.

In [ ]:
class DoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int, alpha: int, dropout: float):
        super().__init__()
        self.in_features = base.in_features
        self.out_features = base.out_features
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        # Freeze Original Weights
        self.weight = nn.Parameter(base.weight.detach().clone(), requires_grad=False)
        self.bias = nn.Parameter(base.bias.detach().clone(), requires_grad=False) if base.bias is not None else None

        # LoRA Parameters
        self.lora_A = nn.Parameter(torch.empty(rank, self.in_features, device=self.weight.device, dtype=self.weight.dtype))
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, rank, device=self.weight.device, dtype=self.weight.dtype))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

        # Magnitude Parameters
        magnitude = torch.linalg.vector_norm(self.weight.float(), ord=2, dim=1).to(self.weight.dtype)
        self.magnitude = nn.Parameter(magnitude)

    def merged_weight(self):
        """Merge weight for inference time"""
        directional = self.weight + self.scaling * (self.lora_B @ self.lora_A)
        norm = torch.linalg.vector_norm(directional.float(), ord=2, dim=1, keepdim=True).to(directional.dtype).clamp_min(1e-6)
        return self.magnitude[:, None] * directional / norm

    def forward(self, x):
        if self.training:
            directional = self.weight + self.scaling * (self.lora_B @ self.lora_A)
            norm = torch.linalg.vector_norm(directional.float(), ord=2, dim=1).to(directional.dtype).clamp_min(1e-6)
            norm_scale = self.magnitude / norm.detach()

            base_out = F.linear(x, self.weight, self.bias)
            lora_out = F.linear(
                F.linear(self.dropout(x), self.lora_A),
                self.lora_B
            ) * self.scaling

            return norm_scale * (base_out + lora_out)
        else:
            return F.linear(self.dropout(x), self.merged_weight(), self.bias)


def replace_target_linears_with_dora(model: nn.Module, target_modules: Iterable[str], rank: int, alpha: int, dropout: float) -> int:
    modules = dict(model.named_modules())
    replacements = []
    target_modules = tuple(target_modules)
    for name, module in modules.items():
        if isinstance(module, nn.Linear) and any(target == name.split('.')[-1] or target in name for target in target_modules):
            parent_name, child_name = name.rsplit('.', 1) if '.' in name else ('', name)
            parent = modules[parent_name] if parent_name else model
            replacements.append((parent, child_name, name))
    for parent, child_name, full_name in replacements:
        old = getattr(parent, child_name)
        setattr(parent, child_name, DoRALinear(old, rank=rank, alpha=alpha, dropout=dropout))
    return len(replacements)


def freeze_all_then_unfreeze_dora(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = False
    for module in model.modules():
        if isinstance(module, DoRALinear):
            module.lora_A.requires_grad = True
            module.lora_B.requires_grad = True
            module.magnitude.requires_grad = True


def trainable_parameter_summary(model: nn.Module) -> Tuple[int, int, float]:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total, 100.0 * trainable / total

print("DoRA modules defined.")

### Smoketest Sanity Check For DoRA

In [ ]:
print("Running DoRALinear startup identity test...")

_test_layer = nn.Linear(7, 4, bias=False)
_wrap_layer = DoRALinear(_test_layer, rank=2, alpha=4, dropout=0.0)
_dummy_input = torch.randn(3, 7)

# Verify outputs match exactly at initialization
with torch.no_grad():
    assert torch.allclose(_test_layer(_dummy_input), _wrap_layer(_dummy_input), atol=1e-5), "Mismatch detected!"

print("SUCCESS: Identity test passed. DoRA starts cleanly.")

## 4. Dataset Loading



In [ ]:
class DreamBoothDataset(Dataset):
    def __init__(self, hf_dataset_id, prompt, size=1024):
        self.dataset = load_dataset(hf_dataset_id, split="train")
        self.prompt = prompt
        self.transform = transforms.Compose([
            transforms.Resize(size, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        image = self.dataset[index]["image"]
        if not image.mode == "RGB":
            image = image.convert("RGB")
        return {"pixel_values": self.transform(image), "prompt": self.prompt}

print(f"Testing dataset loader for: {DATASET_ID}")
_dummy_dataset = DreamBoothDataset(DATASET_ID, INSTANCE_PROMPT)
print(f"Dataset successfully loaded. Total samples: {len(_dummy_dataset)}")

## 5. SDXL Training

In [ ]:
def train_sdxl_dora():
    print("Loading SDXL components into memory...")
    tokenizer_one = AutoTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer", use_fast=False)
    tokenizer_two = AutoTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer_2", use_fast=False)

    text_encoder_one = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder", torch_dtype=WEIGHT_DTYPE).to(DEVICE)
    text_encoder_two = CLIPTextModelWithProjection.from_pretrained(MODEL_ID, subfolder="text_encoder_2", torch_dtype=WEIGHT_DTYPE).to(DEVICE)
    vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=WEIGHT_DTYPE).to(DEVICE)
    unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet", torch_dtype=WEIGHT_DTYPE).to(DEVICE)
    noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

    # Freeze encoders and VAE
    vae.requires_grad_(False)
    text_encoder_one.requires_grad_(False)
    text_encoder_two.requires_grad_(False)

    print("Injecting DoRA modules into UNet...")
    target_modules = ["to_q", "to_k", "to_v", "to_out.0"]
    replaced_count = replace_target_linears_with_dora(unet, target_modules, RANK, ALPHA, DROPOUT)
    freeze_all_then_unfreeze_dora(unet)

    trainable_params, total_params, pct = trainable_parameter_summary(unet)
    print(f"Injection complete! Replaced {replaced_count} target layers.")
    print(f"Trainable params: {trainable_params / 1e6:.2f} M ({pct:.4f}% of total UNet parameters)")

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, unet.parameters()), lr=LR)
    train_dataset = DreamBoothDataset(DATASET_ID, INSTANCE_PROMPT)
    train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

    print(f"Starting training for {STEPS} steps...")
    unet.train()
    global_step = 0
    pbar = tqdm(total=STEPS, desc="Training Progress")

    while global_step < STEPS:
        for batch in train_dataloader:
            if global_step >= STEPS: break

            optimizer.zero_grad()

            pixel_values = batch["pixel_values"].to(DEVICE, dtype=WEIGHT_DTYPE)
            with torch.no_grad(): # Critical for memory saving
                latents = vae.encode(pixel_values).latent_dist.sample()
                latents = latents * vae.config.scaling_factor

            noise = torch.randn_like(latents)
            bsz = latents.shape[0]
            timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=DEVICE).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            prompts = batch["prompt"]
            with torch.no_grad():
                inputs1 = tokenizer_one(prompts, padding="max_length", max_length=77, truncation=True, return_tensors="pt").to(DEVICE)
                embeds1 = text_encoder_one(inputs1.input_ids, output_hidden_states=True)
                hidden1 = embeds1.hidden_states[-2]

                inputs2 = tokenizer_two(prompts, padding="max_length", max_length=77, truncation=True, return_tensors="pt").to(DEVICE)
                embeds2 = text_encoder_two(inputs2.input_ids, output_hidden_states=True)
                hidden2 = embeds2.hidden_states[-2]
                pooled2 = embeds2[0]

                prompt_embeds = torch.cat([hidden1, hidden2], dim=-1)
                add_time_ids = torch.tensor([[1024, 1024, 0, 0, 1024, 1024]], dtype=WEIGHT_DTYPE, device=DEVICE).repeat(bsz, 1)
                added_cond_kwargs = {"text_embeds": pooled2, "time_ids": add_time_ids}

            # Predict noise residual
            model_pred = unet(noisy_latents, timesteps, prompt_embeds, added_cond_kwargs=added_cond_kwargs).sample
            loss = F.mse_loss(model_pred.float(), noise.float(), reduction="mean")

            loss.backward()
            optimizer.step()

            pbar.update(1)
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            global_step += 1

    print("Training phase completed.")
    return unet

# Run the training
trained_unet = train_sdxl_dora()

## 6. Evaluation and Visualization


In [ ]:
print("Preparing Diffusion Pipeline for inference...")

# Load standard SDXL pipeline and swap in our trained UNet
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    unet=trained_unet,
    torch_dtype=WEIGHT_DTYPE
).to(DEVICE)

# Optional: Enable memory optimizations if you are running out of VRAM during inference
# pipe.enable_model_cpu_offload()

eval_prompt = "A photo of sks dog sitting on a snowy mountain, highly detailed, 8k resolution"
print(f"Generating image with prompt: '{eval_prompt}'")

# Execute inference
image = pipe(
    eval_prompt,
    num_inference_steps=30,
    guidance_scale=7.5
).images[0]

# Display the output
plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.axis("off")
plt.title("DoRA Fine-tuned SDXL Output")
plt.show()

# Save the image locally
output_filename = "dora_sdxl_output.png"
image.save(output_filename)
print(f"Image successfully saved as: {output_filename}")